In [4]:
from src import Agent, llm
from src.tools import ddgs_tool
from src.store import Store

store = Store()

In [2]:
LOCATION = 'Гатчинский муниципальный округ (бывш. район) (Ленинградская область)'
YEAR = 2026
INITIAL_PLAN = {
    'социально-экономический анализ': [
        'демография', 
        'производительность труда', 
        'структура и динамика ВГП', 
        'рынок труда', 
        'рынок жилья и коммерческой недвижимости', 
        'бюджетная обеспеченность'
    ],
    'пространственный анализ': [
        'функциональная организация территории', 
        'структура землевладения', 
        'природноэкологический каркас', 
        'архитектурно-градостроительная среда'
    ],
    'транспортный анализ':[
        'городской и внешний пассажирский и грузовой транспорт',
        'парковки',
        'пешеходные зоны',
    ],
    'анализ инженерной инфраструктуры':[
        'водоснабжение и водоотведение', 
        'энергоснабжение', 
        'теплоснабжение',
    ]
}

In [ ]:
GENERATOR_PROMPT = """
Вы агент по поиску информации.
Ответьте основываясь строго на данных из инструментов.
---
ИНСТРУМЕНТЫ:
- ddgs_tool - используй для поиска в интернете.
- search - используй для поиска по документам в векторной базе.
---
ПРАВИЛА РАБОТЫ:
- НЕ ОТВЕЧАЙ на основе общих знаний.
- НЕ ДАВАЙ рекомендаций или комментариев.
- Постарайся МАКСИМАЛЬНО раскрыть тему.
- Подкрепляй факты КОНКРЕТНЫМИ цифрами.
- ИЗБЕГАЙ лишней "воды".
---
КОНТЕКСТ:
- Территория: {location}
- Текущий год: {year}
---
ВОПРОС:
{section}. Рассматриваемые темы: {topics}
"""

CRITIC_PROMPT = """
Вы строгий критик.
Оцените ответ агента и укажите, чего не хватает.
---
КОНТЕКСТ:
- Территория: {location}
- Текущий год: {year}
---
ВОПРОС:
{section}. Рассматриваемые темы: {topics}
"""

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

class GraphState(TypedDict):
    params : dict[str]
    iterations : int
    _iteration : int
    _generator : Agent
    _critic : Agent
    result : str
    critique : str
    
def master(state : GraphState):
    params = state['params']
    return {
        'iterations': state.get('iterations', 3),
        '_iteration': 0,
        '_generator': Agent(GENERATOR_PROMPT.format(**params), tools=[ddgs_tool, store.tool]),
        '_critic': Agent(CRITIC_PROMPT.format(**params)),
    }

def generator(state : GraphState):
    iteration = state['_iteration']
    generator = state['_generator']

    res = generator.invoke(state.get('critique', []))

    return {
        '_iteration' : iteration + 1,
        'result': res
    }

def generator_conditional_edges(state : GraphState):
    iteration = state['_iteration']
    iterations = state['iterations']
    if iteration == iterations:
        return END
    return 'critic'

def critic(state : GraphState):
    critic = state['_critic']
    res = critic.invoke(state['result'])
    return {
        'critique': res
    }


graph = StateGraph(GraphState)

graph.add_node('master', master)
graph.add_node('generator', generator)
graph.add_node('critic', critic)

graph.add_edge(START, 'master')
graph.add_edge('master', 'generator')
graph.add_conditional_edges('generator', generator_conditional_edges, [END, 'critic'])
graph.add_edge('critic', 'generator')

app = graph.compile(debug=True)


In [38]:
app.invoke({'params':{
    'location': LOCATION,
    'year': YEAR,
    'section': 'социально-экономический анализ',
    'topics': ', '.join(INITIAL_PLAN['социально-экономический анализ'])
}, 'iterations': 5})

[values] {'params': {'location': 'Гатчинский муниципальный округ (бывш. район) (Ленинградская область)', 'year': 2026, 'section': 'социально-экономический анализ', 'topics': 'демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность'}, 'iterations': 5}
[updates] {'master': {'iterations': 5, '_iteration': 0, '_generator': <src.agent.Agent object at 0x7fe1329f81f0>, '_critic': <src.agent.Agent object at 0x7fe132a41c90>}}
[values] {'params': {'location': 'Гатчинский муниципальный округ (бывш. район) (Ленинградская область)', 'year': 2026, 'section': 'социально-экономический анализ', 'topics': 'демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность'}, 'iterations': 5, '_iteration': 0, '_generator': <src.agent.Agent object at 0x7fe1329f81f0>, '_critic': <src.agent.Agent object at 0x7fe132a41c90>}
[values] {'messages': []

{'params': {'location': 'Гатчинский муниципальный округ (бывш. район) (Ленинградская область)',
  'year': 2026,
  'section': 'социально-экономический анализ',
  'topics': 'демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность'},
 'iterations': 5,
 '_iteration': 5,
 '_generator': <src.agent.Agent at 0x7fe1329f81f0>,
 '_critic': <src.agent.Agent at 0x7fe132a41c90>,
 'result': '### Социально-экономический анализ Гатчинского муниципального округа на 2026 год\n\n#### 1. Демография\n- **Население**: В 2026 году численность населения Гатчинского округа составляет около 100,000 человек. Основные демографические показатели включают уровень рождаемости, который составляет 10,5 на 1000 человек, и уровень смертности — 12,0 на 1000 человек.\n- **Возрастная структура**: Примерно 20% населения составляет молодежь (до 30 лет), 60% — трудоспособное население (30-60 лет), и 20% — пожилые люди (старше 60 лет).\n- **М